# Análise comparativa de quatro lojas

**Objetivo:** comparar indicadores descritivos de quatro bases de compras e produzir evidências para uma decisão comercial.

> Os dados são públicos e pertencem ao desafio de Data Science da Alura. Este notebook distingue soma dos preços registrados, avaliação média e frete médio. Não trata a soma de preços como faturamento comprovado, pois a base não informa explicitamente a quantidade de unidades por registro nem a regra contábil de receita.

## 1. Bibliotecas e fonte dos dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

BASE_URL = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science/refs/heads/main/base-de-dados-challenge-1/"
URLS = {f"Loja {i}": f"{BASE_URL}loja_{i}.csv" for i in range(1, 5)}

lojas = {nome: pd.read_csv(url) for nome, url in URLS.items()}
print("Bases carregadas:", ", ".join(lojas))


## 2. Entendimento e qualidade dos dados

In [ ]:
for nome, df in lojas.items():
    print(f"\\n{nome}: {df.shape[0]} linhas e {df.shape[1]} colunas")
    display(df.head(3))
    display(df.dtypes.to_frame("tipo"))


In [ ]:
# Verificar campos esperados, ausências e duplicidades
colunas_esperadas = {
    "Produto", "Categoria do Produto", "Preço", "Frete",
    "Data da Compra", "Vendedor", "Local da compra",
    "Avaliação da compra", "Tipo de pagamento",
    "Quantidade de parcelas", "lat", "lon"
}

qualidade = []
for nome, df in lojas.items():
    faltantes_colunas = sorted(colunas_esperadas - set(df.columns))
    qualidade.append({
        "Loja": nome,
        "Registros": len(df),
        "Colunas ausentes": ", ".join(faltantes_colunas) if faltantes_colunas else "Nenhuma",
        "Células ausentes": int(df.isna().sum().sum()),
        "Linhas duplicadas": int(df.duplicated().sum())
    })

qualidade_df = pd.DataFrame(qualidade)
display(qualidade_df)


In [ ]:
# Padronização dos tipos usados nos indicadores
for nome, df in lojas.items():
    for coluna in ["Preço", "Frete", "Avaliação da compra"]:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")
    # A base usa dia/mês/ano; declarar o formato evita interpretar datas ambíguas como mês/dia/ano.
    df["Data da Compra"] = pd.to_datetime(
        df["Data da Compra"], format="%d/%m/%Y", errors="coerce"
    )

datas_invalidas = {
    nome: int(df["Data da Compra"].isna().sum())
    for nome, df in lojas.items()
}
print("Conversão concluída. Datas não interpretadas por loja:", datas_invalidas)
print("Verifique também os valores ausentes reportados na etapa de qualidade antes de interpretar os indicadores.")


## 3. Indicadores comparativos

A soma de `Preço` representa a soma dos valores encontrados nessa coluna. Ela só deve ser chamada de faturamento se a definição da base confirmar que cada registro corresponde ao valor integral de uma venda.

In [ ]:
resumo = []
for nome, df in lojas.items():
    resumo.append({
        "Loja": nome,
        "Registros": len(df),
        "Soma dos preços registrados (R$)": df["Preço"].sum(),
        "Avaliação média (0 a 5)": df["Avaliação da compra"].mean(),
        "Frete médio (R$)": df["Frete"].mean()
    })

indicadores = pd.DataFrame(resumo).set_index("Loja")
display(indicadores.round(2))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

indicadores["Soma dos preços registrados (R$)"].sort_values(ascending=False).plot(
    kind="bar", ax=axes[0], title="Soma dos preços registrados"
)
axes[0].set_ylabel("R$")
axes[0].tick_params(axis="x", rotation=0)

indicadores["Avaliação média (0 a 5)"].sort_values(ascending=False).plot(
    kind="bar", ax=axes[1], title="Avaliação média"
)
axes[1].set_ylabel("Nota (0 a 5)")
axes[1].tick_params(axis="x", rotation=0)

indicadores["Frete médio (R$)"].sort_values().plot(
    kind="bar", ax=axes[2], title="Frete médio"
)
axes[2].set_ylabel("R$")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


## 4. Distribuição por categoria

A contagem abaixo mede o número de registros em cada categoria, não unidades vendidas.

In [ ]:
categorias = pd.concat(
    [df.assign(Loja=nome) for nome, df in lojas.items()],
    ignore_index=True
)
contagem_categoria = (
    categorias.groupby(["Loja", "Categoria do Produto"])
    .size().rename("Registros").reset_index()
)
tabela_categoria = contagem_categoria.pivot(
    index="Categoria do Produto", columns="Loja", values="Registros"
).fillna(0)
display(tabela_categoria.astype(int))

tabela_categoria.plot(kind="bar", figsize=(14, 6))
plt.title("Registros por categoria e loja")
plt.xlabel("Categoria")
plt.ylabel("Número de registros")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 5. Produtos com maior frequência de registros

In [ ]:
for nome, df in lojas.items():
    frequencia = df["Produto"].value_counts()
    print(f"\\n{nome} — 5 produtos com mais registros")
    display(frequencia.head(5).rename("Número de registros").to_frame())


## 6. Exploração geográfica

O gráfico mostra a distribuição das coordenadas registradas. Sem uma validação geográfica adicional e dados socioeconômicos externos, não é possível inferir poder aquisitivo ou explicar causalidade.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for nome, df in lojas.items():
    ax.scatter(df["lon"], df["lat"], s=10, alpha=0.35, label=nome)
ax.set_title("Distribuição geográfica dos registros")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.tight_layout()
plt.show()


## 7. Síntese executiva

In [ ]:
maior_soma = indicadores["Soma dos preços registrados (R$)"].idxmax()
melhor_avaliacao = indicadores["Avaliação média (0 a 5)"].idxmax()
menor_frete = indicadores["Frete médio (R$)"].idxmin()

print(f"Maior soma dos preços registrados: {maior_soma} (R$ {indicadores.loc[maior_soma, 'Soma dos preços registrados (R$)']:,.2f})")
print(f"Maior avaliação média: {melhor_avaliacao} ({indicadores.loc[melhor_avaliacao, 'Avaliação média (0 a 5)']:.2f}/5)")
print(f"Menor frete médio: {menor_frete} (R$ {indicadores.loc[menor_frete, 'Frete médio (R$)']:.2f})")
print("\\nConclusão: os indicadores apontam lideranças diferentes. A decisão depende da prioridade comercial; não há uma vencedora absoluta em todos os critérios.")


## 8. Limitações e próximos passos

- A base não possui campo explícito de unidades vendidas; contagens representam registros.
- A soma de `Preço` não deve ser denominada faturamento sem confirmação da granularidade e da regra de negócio.
- A análise geográfica é descritiva e não demonstra causalidade.
- Próximos passos: investigar datas e sazonalidade, validar outliers, documentar regras de negócio e, se houver dados adequados, integrar indicadores de margem e quantidade.